# Session 4: From Batch to Intraday Operations

We have a validated Cobb-Douglas engine, EWLS-adapted SIM parameters, and a paper-trading account. The question is no longer *does the strategy work?*; it is *can we operate it on a desk-realistic cadence?* In this session, we move the engine from a once-daily rebalance to a **30-minute intraday cadence** during market hours, wrap it in a two-tier compliance system, and run a 7pm desk-review meeting in which we adjudicate the day's flagged trades and sign off on tomorrow's opening posture. Two AI layers are now live in production.

* **The autonomous decision agent** is the engine built across Sessions 2 and 3. Each 30-minute fire pulls the latest bar, updates SIM parameters via [`ewls_update!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#ewls_update!), runs the regime-aware allocator, and proposes trades.
* **A two-tier compliance system** wraps each proposal: trades that clear pre-approved gates auto-execute against Alpaca paper, trades that fail at least one gate route to a queue for human review at the evening desk meeting.

The session treats the engine, the queue, and the evening sign-off as one operating system. Maya Chen runs it; Lou audits it; the cron schedule is the heartbeat.

> __Learning Objectives:__
>
> By the end of this session, we will be able to:
> * __Run the engine on a 30-minute intraday cadence:__ Convert the EWLS half-life and $\Delta t$ from daily to bar units via [`intraday_half_life`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life) and [`intraday_dt`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt), and recognize that the EWLS and bandit framework is bar-interval-agnostic.
> * __Split proposed trades into auto-execute and queue:__ Use [`gate_check`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#gate_check) and [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades) so that small, low-risk trades clear automatically while large or news-flagged trades route to the human-review queue.
> * __Adjudicate the queue and sign tomorrow's ticket:__ Walk a day's [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) list, record decisions as [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) artifacts, and commit the next-day allocation as a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) consumed by tomorrow's 9:35am cron.

Let's run the engine.

___

## Examples

The example notebooks below accompany this lecture. They split into __core__ examples that we run live alongside the lecture, and __optional__ extension material we can study afterward at our own pace.

### Core (tonight)

These three notebooks carry the session's intraday operating loop: the cron-driven engine that fires every 30 minutes during market hours, the evening desk-review notebook that adjudicates today's queued trades, and the sign-off notebook that commits tomorrow's opening posture as a real next-morning order.

> __Run live with the lecture:__
>
* [▶ Let's walk the cron-driven intraday engine](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb). We step through a single 30-minute fire of `production_runner.jl`: bar fetch, EWLS update, allocator, gate split into auto-execute and queue, and audit-tape append. Then we load today's tape and read what the engine actually did.
* [▶ Let's run the evening desk review on today's tape](eCornell-AI-Finance-S4-Example-Core-EveningReview-May-2026.ipynb). We open today's intraday tape, score the day on P&L attribution, EWLS estimate drift, and bandit pulls per regime, then walk the queue trade-by-trade and record adjudications as signed-decision artifacts.
* [▶ Let's refresh the news, recompute the opening allocation, and sign tomorrow's ticket](eCornell-AI-Finance-S4-Example-Core-TomorrowsTicket-May-2026.ipynb). We trigger a live news refresh in front of the class, run [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) against tonight's parameters, and commit a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) that the May 12 9:35am cron will submit to Alpaca paper.

### Optional (self-study)

This notebook goes deeper on the news ingestion pipeline that backs the intraday engine. It is not required for the evening desk review.

> __Extension material (self-study):__
>
* [▶ Let's open the AI sentiment pipeline](eCornell-AI-Finance-S4-Example-Optional-AISentimentDeepDive-May-2026.ipynb). We generate the synthetic news corpus, score it via [`score_news_with_claude!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#score_news_with_claude!), estimate news loadings $\nu_i$ via [`estimate_sim_with_news`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#estimate_sim_with_news), and trace how a single high-severity headline propagates from the news scorer to the engine's compliance queue.
>
> Study this when we want to see the mechanics behind the news flag that triggers a queue routing in the Core notebooks.

___

## From Batch to Intraday

In Session 3, the engine fired once per trading day at the close. That cadence is fine for backtesting but does not match how an institutional desk operates: parameters drift through the trading day, news arrives at all hours, and a once-a-day allocation cannot react to a mid-morning regime shift. We move to a **30-minute intraday cadence** for Session 4.

> __Why intraday is a parameter change, not a redesign:__
>
> The EWLS recursion is a recursive least squares estimator with exponential forgetting. Its update rule depends on the *number of observations*, not on the calendar interval between them. The same is true of the epsilon-greedy bandit: pulls update arm means in constant time per observation. Both algorithms are **bar-interval-agnostic**. Moving from daily to 30-minute bars changes only three knobs: the time step $\Delta t$, the EWLS half-life expressed in bars, and the volatility annualization factor.

Let $B$ denote the number of regular-session bars per trading day for a given bar interval, computed from the 6.5-hour US equity session as $B = \lfloor 390 / \text{bar minutes} \rfloor$. The annualized time step and the calendar-invariant half-life follow:

$$\boxed{\Delta t \;=\; \frac{1}{252 \cdot B}, \qquad h_{\text{bars}} \;=\; h_{\text{calendar days}} \cdot B \quad\blacksquare}$$

The companion script implements these as [`bars_per_day`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#bars_per_day), [`intraday_dt`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt), and [`intraday_half_life`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life). The engine's TOML configuration stores `half_life_calendar_days` (a property a senior PM can reason about) and the runner translates it to `h_{\text{bars}}` at startup, so changing `bar_minutes` in config does not require re-tuning anything else.

This is an *online* algorithm: each cron fire receives the latest bar, updates state, decides, and persists. No batch step. The mechanics are compact enough to state as pseudocode directly.

### Algorithm: Intraday Engine Step (Online)

__Initialize__: Given a ticker universe $\mathcal{U} = \{1, \ldots, K\}$, a bar interval `bar_minutes`, an EWLS state $\{\hat\alpha_i, \hat\beta_i, \hat\sigma_{\varepsilon,i}\}_{i \in \mathcal{U}}$ seeded from a frozen calibration, a compliance gate config $\mathcal{G}$, a rolling intraday market-price buffer $\mathbf{S}_{\text{intra}}$ (reset each trading day), and an Alpaca paper client, set $\Delta t \gets$ [`intraday_dt(bar_minutes)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt) and $h_{\text{bars}} \gets$ [`intraday_half_life(h_{\text{cal}}, bar\_minutes)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life).

On each fire at time $t$ __do__:

1. Fetch new bars since the last seen timestamp; if none, skip.
2. For each new bar at timestamp $t_b$, compute the per-bar growth $g_{m, t_b} \gets \frac{1}{\Delta t} \log(S_{t_b} / S_{t_b - 1})$ and call [`ewls_update!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#ewls_update!) per ticker.
3. Recompute sentiment and effective lambda from $\mathbf{S}_{\text{intra}}$, then bin into a regime $\rho_t \in \{\text{bear}, \text{neutral}, \text{bull}\}$.
4. Run the allocator at the regime-appropriate elasticity $\eta_t$ to obtain target weights $\mathbf{w}^\star_t$.
5. Compute proposed trades $\Delta \mathbf{n}_t$ from current positions, prices, and $\mathbf{w}^\star_t$, with per-trade dollar value and post-trade weight.
6. Join per-ticker news severity from the latest hourly news artifact.
7. Call [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades) with $\Delta \mathbf{n}_t$, the news-severity snapshots, and $\mathcal{G}$ to obtain:
    - $\mathcal{A}_t$: trades that clear all gates (auto-execute via Alpaca).
    - $\mathcal{Q}_t$: trades that fail at least one gate (append to today's queue file as [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem)).
8. Submit each $a \in \mathcal{A}_t$ to Alpaca paper; append each $q \in \mathcal{Q}_t$ to the queue.
9. Append the fire's audit entry (sentiment, $\lambda$, regime, $\eta$, target weights, $|\mathcal{A}_t|$, $|\mathcal{Q}_t|$) to today's tape.

__Output__: Persist the updated state and exit. The next cron fire picks up from the persisted state.

The same step runs 14 times per trading day. The 16:00 close fire additionally invokes [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) to write a [`MyTomorrowsTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyTomorrowsTicket) that the evening review notebook will load.

___

## The Two-Mode Cron: Auto-Execute and Queue

Each fire produces a basket of proposed trades. We do not auto-execute every proposal: a trade large enough to move the portfolio's risk profile materially deserves human review even if the engine likes it. The cron splits proposals into two piles via [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades).

> __Per-trade and portfolio-level gates:__
>
> A trade clears the gates and auto-executes when **all** of the following hold; any failure routes the trade to the queue.

| Gate | Scope | Threshold | Why It Exists |
|------|-------|-----------|---------------|
| **Concentration cap** | per-trade | post-trade weight $w_i$ in any single ticker $\le c_{\max}$ | A single-name bet beyond the cap is a portfolio-level decision, not a model decision. |
| **Position size limit** | per-trade | $\lvert\text{trade dollar value}\rvert \le \text{size}_{\max}$ | Caps the dollar magnitude of any single click. Cheap to enforce, cheap to override. |
| **News severity** | per-trade | news-severity score for the ticker $\le s_{\max}$ | If a high-severity headline lands during the fire, the engine cannot judge it. The PM can. |
| **Turnover limit** | portfolio-level | $\sum_i \lvert\text{trade dollar value}_i\rvert \,/\, V_t \le \tau_{\max}$ | Caps per-fire turnover against portfolio value $V_t$. If exceeded, **all** proposed trades route to the queue (not just the largest). |

Threshold values live in the `[Compliance]` section of `production-config.toml`. We tune them so the queue is non-empty most days: too lax and the evening review meeting has nothing to do; too tight and the queue floods and the engine effectively does nothing automatically. The companion examples take a concrete tuning pass.

Trades that clear get submitted via the Alpaca SDK. Trades that fail get serialized as [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) records (with the engine snapshot stamped in for review context) and appended to today's queue file via [`append_queue_item!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#append_queue_item!).

___

## The Hourly News Pipeline

News flow is the third cron on the schedule. Where the engine fires every 30 minutes, the news scorer fires hourly: each fire pulls headlines, scores them via Claude, aggregates per-ticker sentiment and severity, and writes an artifact at `data/news/news-YYYY-MM-DD-HH.jld2`. The next engine fire reads the latest artifact and joins severity onto each proposed trade.

> __Why hourly, not per-bar:__
>
> Calling Claude on every 30-minute engine fire would double the LLM cost and offer little marginal information; mainstream news cycles change on the order of an hour, not a half hour. Hourly cadence keeps the news signal fresh enough to flag a trade in the same fire that an event lands in, while keeping cost predictable.

The source backing the pipeline is configurable via `config/news-source.toml`. The lecture-time discussion uses three modes: `synthetic` (drives the cron from an on-disk corpus generated by [`simulate_news_corpus`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#simulate_news_corpus); the May 5 to May 10 cron operates on this), `newsapi` (pulls real headlines from a third-party news API), and `anthropic_web` (uses Anthropic's web fetch tool). The class-night demo on May 11 switches to a real-source mode for one fire so the class watches a real headline produce a real sentiment shift in real time.

Every news artifact carries the same shape regardless of source: a list of [`MyNewsItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) records with `claude_score` populated, plus per-ticker aggregates `sentiment` and `severity`. The runner's join is identical between synthetic and real modes, so swapping sources is a one-line config change.

___

## The Evening Desk Review

By 4pm close, the day has produced three artifacts: an intraday tape (one entry per fire), a queue (one entry per flagged trade), and tomorrow's ticket (the engine's proposed opening allocation). The evening desk review reads all three.

> __What the meeting is for:__
>
> The engine has already executed the easy decisions during the day. The meeting exists to handle the *hard* decisions: the trades whose magnitude, concentration, or news context puts them in territory where a model cannot adjudicate alone. Maya works the queue trade-by-trade with the engine snapshot in front of her, decides approve / reject / modify, records a rationale, and signs each decision. The signed decisions are what the next morning's cron actually does.

Each queue item carries its full decision context: the timestamp, the ticker and side, the gate violations that routed it, and an `engine_snapshot::Dict{String,Any}` with the EWLS state, the sentiment, the lambda, the regime, and the news-source path active at the fire. The meeting's deliverable is a vector of [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) records, one per queue item, written to `data/decisions/decisions-YYYY-MM-DD.jld2` via [`save_signed_decisions!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_decisions!).

Three adjudication actions are supported:
* `:approve` (sign the trade as-is; tomorrow's morning cron submits the original quantity)
* `:reject` (sign the trade as cancelled; tomorrow's morning cron skips it)
* `:modify` (sign a new quantity; tomorrow's morning cron submits at the modified size)

The meeting is also where intraday performance gets reviewed: P&L attribution by hour, EWLS estimate drift visualized against the prior-day end-of-day estimates, bandit pulls per regime, and any escalation events the runner logged. Lou audits the rationale field on every decision; an undocumented decision is a finding.

___

## Tomorrow's Ticket and Sign-Off

After the queue is worked, attention shifts to tomorrow's opening posture. The 16:00 close fire wrote `data/tickets/ticket-YYYY-MM-DD.jld2` (where the date is the *next* trading day) via [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket). The ticket carries the engine's recommended target weights and the trade list needed to move from today's closing positions to those weights, plus the sentiment snapshot, news flags, the elasticity used, and the regime label.

> __The sign-off step:__
>
> The class refreshes the news pipeline one more time at 7pm against tonight's headlines (the only real-source fire in the week if we are in synthetic mode for May 5 to 10). It then re-runs the allocator with the refreshed sentiment, compares the resulting ticket to the 16:00-generated ticket, and decides whether to commit the original ticket, modify it, or reject specific trades. The committed artifact is a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) wrapping the original ticket plus any modifications, signed by Maya, persisted via [`save_signed_ticket!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_ticket!).

The next-morning 9:35am cron runs `production_runner.jl --mode=execute_signed_ticket`, loads the signed ticket via [`load_signed_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#load_signed_ticket), applies any sign-off modifications, and submits the resulting orders to Alpaca paper. This closes the loop: the evening's human-in-the-loop decision is mechanically committed to the next morning's market open.

___

## Risks and Limits

The intraday loop closes a real human-machine cycle, but several failure modes deserve explicit treatment before we hand the engine the keys.

* **Overnight gaps.** A signed ticket sized at last night's close prices submits at this morning's open prices. If the underlying gaps overnight (earnings, macro print, geopolitical event), the realized fill differs from the engine's intent. The compliance queue does not see overnight risk; the position-size and concentration gates do, but only on the *signed* quantities, not on the gap-adjusted ones. Mitigation lives in setting the position-size limit conservative enough that an overnight gap on any single name is bounded.
* **News API outage at 7pm.** If the real-source news fire fails (rate limit, network, vendor downtime), the sign-off notebook falls back to the synthetic corpus rather than blocking the meeting. The fallback path is documented in the TomorrowsTicket notebook; the lecture is honest that *some* live demos are not actually live.
* **Stale EWLS state on long weekends.** The EWLS half-life is calendar-day-anchored, so a three-day weekend is no different from a regular weekend. But if the cron is paused for a full week (vacation, infrastructure work), the persisted state's `last_bar_timestamp` falls far behind and the next fire fetches a long bar history. The runner handles this without complaint, but the EWLS estimates absorb a week of update steps in a single fire and can move materially. We document a rule of thumb: any pause longer than a week, re-seed EWLS from a fresh calibration before resuming.
* **Compliance gate hyperparameter drift.** Tuning the gates so the queue is non-empty most days is a live calibration, not a one-time setting. As the portfolio grows, the position-size limit and concentration cap need to scale, or the queue empties and the meeting becomes a rubber-stamp.

These are the lecture's scope. Production deployment beyond paper trading invites additional concerns (regulatory reporting, broker outages, fat-finger protections, time-of-day liquidity bias) that we leave to follow-up engagement.

___

## Summary

In this session, we moved the engine from a daily backtest cadence to a 30-minute intraday operating loop, wrapped each proposal in a two-tier compliance system, and structured the human-in-the-loop step as an evening desk-review meeting that signs off on the next morning's open.

> __Key Takeaways:__
>
> * __EWLS and the bandit are bar-interval-agnostic:__ The same online estimators that powered the daily-cadence engine in Session 3 run on 30-minute bars unchanged; only $\Delta t$, the half-life unit, and volatility annualization need to be retuned. We can move the cadence faster or slower without redesigning the algorithm.
> * __The queue is where human judgment lives:__ Auto-executing every proposal is reckless; routing every proposal to a human is paralysis. Tightening the gates so the queue is non-empty most days produces a steady stream of decisions where the model's view and the PM's view actually diverge, and the meeting is where that disagreement gets adjudicated.
> * __The signed ticket closes the loop:__ Tonight's sign-off becomes tomorrow's submitted order. The evening review is not a documentation exercise; it is the production system's commit step, and the next morning's cron is what makes it real.

This is the final session of the course. The full pipeline (SIM estimation in Session 1, adaptive allocation in Session 2, online learning and validation in Session 3, intraday production with evening sign-off in Session 4) is now end-to-end.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The production system described is a pedagogical tool using paper trading and synthetic news. Real-world deployment requires regulatory approval, compliance review, operational infrastructure, and risk management beyond the scope of this course.

___